# 5장. 크기를 키우며 손실 재기

4장에서 거른 위키백과 문장으로, 모델의 크기(파라미터 수)와 학습한 토큰 수를 바꿔 가며 검증 손실을 잰다. 작은 규모에서도 손실이 크기와 토큰 수의 거듭제곱 꼴로 내려가는지, 그 식으로 더 큰 모델을 미리 맞힐 수 있는지 본다.

**준비:** 한국어 위키백과 덤프의 첫 조각 `kowiki-latest-pages-articles1.xml-p1p82407.bz2`(4장과 같은 파일)를 왼쪽 파일 창에 올리고, 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 80분쯤 걸렸다.

## 5.1 4장의 거른 문장으로 학습 자료 만들기

In [ ]:
import bz2, re, math, time, os, sys, random, json, platform
from collections import Counter
from xml.etree.ElementTree import iterparse
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
os.environ["RAYON_NUM_THREADS"] = "2"
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
torch.set_num_threads(2)

pages = []
with bz2.open("kowiki-latest-pages-articles1.xml-p1p82407.bz2", "rb") as f:
    for _, el in iterparse(f, events=("end",)):
        if el.tag.endswith("}page"):
            if el.findtext("{*}ns") == "0" and el.find("{*}redirect") is None:
                pages.append((el.findtext("{*}title"), el.findtext("{*}revision/{*}text") or ""))
            el.clear()
print(len(pages))
print(sys.version.split()[0], torch.__version__, tokenizers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
# 4장의 clean(), sentences()와 고친 품질 규칙을 그대로 쓴다(근사 중복 제거는 생략한다)
def strip_nested(text, left, right):
    """가장 안쪽의 left ... right를 지우기를 더 지울 것이 없을 때까지 한다(틀 {{ }}, 표 {| |})."""
    inner = re.compile(re.escape(left) + r"(?:(?!" + re.escape(left) + r").)*?" + re.escape(right), re.S)
    while True:
        new = inner.sub("", text)
        if new == text:
            return new
        text = new

def clean(wiki):
    """위키 문법을 걷어 내고 문단(줄) 목록을 돌려준다."""
    t = re.sub(r"<!--.*?-->", "", wiki, flags=re.S)                       # 주석
    t = re.sub(r"<ref[^>]*/>", "", t)                                      # 각주
    t = re.sub(r"<ref[^>]*>.*?</ref>", "", t, flags=re.S)
    t = strip_nested(t, "{{", "}}")                                        # 틀(정보 상자 등)
    t = strip_nested(t, "{|", "|}")                                        # 표
    t = re.sub(r"<(math|gallery|timeline|score|syntaxhighlight|source|pre|code)[^>]*>.*?</\1>", "", t, flags=re.S | re.I)
    t = re.sub(r"<[^>]+>", "", t)                                          # 남은 HTML 태그
    t = re.sub(r"\[\[(파일|File|그림|Image|분류|Category):[^\[\]]*(\[\[[^\]]*\]\][^\[\]]*)*\]\]", "", t, flags=re.I)
    t = re.sub(r"\[\[([^\]|]*)\|([^\]]*)\]\]", r"\2", t)                   # [[문서|보이는 글]] → 보이는 글
    t = re.sub(r"\[\[([^\]]*)\]\]", r"\1", t)                              # [[문서]] → 문서
    t = re.sub(r"\[https?://[^\s\]]+ ([^\]]*)\]", r"\1", t)                # [주소 글] → 글
    t = re.sub(r"\[https?://[^\]]*\]", "", t)
    t = re.sub(r"'{2,}", "", t)                                            # 굵게·기울임
    t = re.sub(r"__[A-Z]+__", "", t)
    out = []
    for line in t.split("\n"):
        line = line.strip()
        if not line or line.startswith(("=", "|", "!", "{", "}")):          # 제목, 표의 나머지
            continue
        line = re.sub(r"^[*#:;]+\s*", "", line)                            # 목록 기호
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            out.append(line)
    return out

def sentences(paragraph):
    return [s.strip() for s in re.split(r"(?<=[.!?。])\s+", paragraph) if s.strip()]

hangul = lambda s: sum("가" <= ch <= "힣" for ch in s)
RULES = [lambda s: len(s) < 10,
         lambda s: hangul(s) / max(1, sum(ch.isalpha() for ch in s)) < 0.5,
         lambda s: bool(re.search(r"\(\s*[,;]?\s*\)|,\s*,|\(\s*,", s)),
         lambda s: not re.search(r"[.!?。」』\"'”)]$", s)]
sents = [(d, s) for d, (_, text) in enumerate(pages) for p in clean(text) for s in sentences(p)]
seen, good = set(), []
for d, s in sents:
    if s not in seen:
        seen.add(s)
        if not any(rule(s) for rule in RULES):
            good.append((d, s))
held = set(random.Random(0).sample(range(len(pages)), len(pages) // 10))   # 4장과 같은 검증용 문서
train_pool = [s for d, s in good if d not in held]
val_pool = [s for d, s in good if d in held]
print(len(sents), len(good), len(train_pool), len(val_pool))

In [ ]:
pick = random.Random(0)
letters = {ch for ch, n in Counter(ch for s in pick.sample(train_pool, 300000) for ch in s).items() if n >= 5}
UNK_CH, SP = "□", "▁"
to_text = lambda t: "".join(ch if ch in letters else UNK_CH for ch in t[:64]).replace(" ", SP)
tok = Tokenizer(models.BPE(unk_token=UNK_CH))
tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
tok.train_from_iterator([to_text(t) for t in pick.sample(train_pool, 200000)], trainers.BpeTrainer(
    vocab_size=8000, show_progress=False, initial_alphabet=sorted(letters - {" "} | {SP, UNK_CH}),
    special_tokens=["<패딩>", "<시작>", "<끝>"]))
PAD, START, END, VOCAB = 0, 1, 2, tok.get_vocab_size()

def to_tensor(texts):
    rows = [[START] + e.ids + [END] for e in tok.encode_batch([to_text(t) for t in texts])]
    X = torch.zeros(len(rows), 66, dtype=torch.long)
    for i, r in enumerate(rows):
        X[i, :len(r)] = torch.tensor(r)
    return X

order = pick.sample(train_pool, len(train_pool))         # 한 번 섞은 순서를 모든 학습이 함께 쓴다
X_dev = to_tensor(order[:5000])                            # 학습률을 고르는 개발 문장(학습에 쓰지 않는다)
X_pool = to_tensor(order[5000:])
X_val = to_tensor(pick.sample(val_pool, 10000))            # 검증 문장(검증용 문서에서)
cum = torch.cumsum((X_pool != PAD).sum(dim=1) - 1, dim=0)  # 앞에서부터 쌓은 목표 토큰 수
first = lambda D: X_pool[:int(torch.searchsorted(cum, D)) + 1]   # 목표 토큰이 D개가 되는 앞부분
print(VOCAB, len(letters), len(X_pool), int(cum[-1]), round(cum[-1].item() / len(X_pool), 2))

In [ ]:
class Block(nn.Module):
    """2장의 블록과 같다. 어텐션은 3장 3.5절에서 맞춰 본 scaled_dot_product_attention으로 계산한다."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        B, L, D = x.shape
        split = lambda t: t.view(B, L, self.heads, D // self.heads).transpose(1, 2)
        h = self.norm1(x)
        a = F.scaled_dot_product_attention(split(self.to_q(h)), split(self.to_k(h)), split(self.to_v(h)), is_causal=True)
        x = x + self.out(a.transpose(1, 2).reshape(B, L, D))
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    def __init__(self, dim, layers, heads, size=8000, length=66):
        super().__init__()
        self.table = nn.Embedding(size, dim)
        self.where = nn.Embedding(length, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids):
        x = self.table(ids) + self.where(torch.arange(ids.shape[1]))
        for block in self.blocks:            # 패딩은 문장 뒤에만 있어서, 인과 마스크만으로 실제 토큰이 패딩을 보지 않는다
            x = block(x)
        return self.head(self.norm(x))

SIZES = {"S1": (32, 1, 2), "S2": (48, 2, 4), "S3": (64, 2, 4), "S4": (96, 3, 4), "S5": (128, 4, 4), "S6": (192, 4, 6)}
def counts(net):
    body = sum(p.numel() for n, p in net.named_parameters() if n.startswith(("blocks.", "norm.")))
    return body, sum(p.numel() for p in net.parameters())
for name, (d, l, h) in SIZES.items():
    body, total = counts(LanguageModel(d, l, h))
    print(f"{name}  벡터 {d:>3}  블록 {l}  머리 {h}  몸통 {body:>9,}  전체 {total:>9,}  몸통 비율 {body / total:.1%}")

In [ ]:
def fit(net, X, lr, seed):
    """한 번만 읽고 지나간다(1에폭). AdamW, 배치 64, 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    shuffle = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))
    steps = math.ceil(len(X) / 64)
    warm = max(1, steps // 10)
    net.train()
    for step in range(1, steps + 1):
        ids = X[shuffle[(step - 1) * 64:step * 64]]
        ids = ids[:, :int((ids != PAD).sum(dim=1).max())]
        for group in optimizer.param_groups:
            group["lr"] = lr * min(1, step / warm) * 0.5 * (1 + math.cos(math.pi * step / steps))
        out = net(ids[:, :-1])
        loss = F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1), ignore_index=PAD)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return net

def loss_of(net, X):
    """목표 토큰 하나당 평균 손실(나트)."""
    net.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for k in range(0, len(X), 500):
            ids = X[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                                     ignore_index=PAD, reduction="sum").item()
            n += (ids[:, 1:] != PAD).sum().item()
    return total / n

runs = {}
def run(size, D, lr, seed=0):
    d, l, h = SIZES[size]
    torch.manual_seed(seed)
    net = LanguageModel(d, l, h)
    lap = time.perf_counter()
    fit(net, first(D), lr, seed)
    sec = time.perf_counter() - lap
    runs[size, D, lr, seed] = {"dev": loss_of(net, X_dev), "val": loss_of(net, X_val), "sec": sec}
    return runs[size, D, lr, seed]

## 5.2 크기마다 학습률 고르기

In [ ]:
RATES = [0.001, 0.003, 0.01, 0.03]
D0 = 2_000_000
print("크기  " + "  ".join(f"{lr:>7}" for lr in RATES) + "   (개발 문장의 손실, 토큰 200만 개)")
for size in SIZES:
    got = [run(size, D0, lr)["dev"] for lr in RATES]
    print(f"{size:<4}  " + "  ".join(f"{v:7.4f}" for v in got), flush=True)
best = {s: min(RATES, key=lambda lr: runs[s, D0, lr, 0]["dev"]) for s in SIZES}
print(best)

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
SHADE = ["#B9CCF3", "#8FAEEB", "#5F8BE2", "#2F6FDB", "#1F4FA8", "#13306A"]       # 작은 모델일수록 옅게
from matplotlib.ticker import FuncFormatter, NullFormatter
def man(v, _=None):
    """축 눈금을 10000 → 1만, 2000000 → 200만, 100000000 → 1억처럼 쓴다."""
    return f"{v / 10**8:,g}억" if v >= 10**8 else f"{v / 10**4:,g}만"
def param_axis(ax):
    ax.xaxis.set_major_formatter(FuncFormatter(man)); ax.xaxis.set_minor_formatter(NullFormatter())

fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.6), dpi=260)
ax = axes[0]
for i, s in enumerate(SIZES):
    ys = [runs[s, D0, lr, 0]["dev"] for lr in RATES]
    ax.plot(RATES, ys, "-o", color=SHADE[i], lw=1.4, ms=3.6, label=f"{s}(몸통 {counts(LanguageModel(*SIZES[s]))[0]:,})")
    ax.plot([best[s]], [min(ys)], "*", color=ORANGE, ms=10, zorder=5)
ax.set_xscale("log"); ax.set_xticks(RATES); ax.set_xticklabels([str(r) for r in RATES])
ax.set_xlabel("학습률(로그 눈금)"); ax.set_ylabel("개발 문장의 손실(나트)")
ax.set_title("① 크기마다 학습률을 바꿔 본 손실(주황 별: 가장 낮은 학습률)", loc="left", fontsize=8.6, color="#374151")
ax.set_ylim(top=8.05)                                                # 위쪽 빈 곳에 범례를 둔다
ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", columnspacing=0.8, handletextpad=0.3, handlelength=1.4)
ax = axes[1]
xs = [counts(LanguageModel(*SIZES[s]))[0] for s in SIZES]
ax.plot(xs, [runs[s, D0, 0.01, 0]["val"] for s in SIZES], "-o", color=RED, lw=1.4, ms=3.6, label="모든 크기에 학습률 0.01")
ax.plot(xs, [runs[s, D0, best[s], 0]["val"] for s in SIZES], "-o", color=GREEN, lw=1.4, ms=3.6, label="크기마다 고른 학습률")
for x, s in zip(xs, SIZES):
    ax.annotate(s, (x, runs[s, D0, best[s], 0]["val"]), xytext=(0, -13), textcoords="offset points", ha="center", fontsize=8, color="#374151")
ax.set_xscale("log"); param_axis(ax); ax.set_xlabel("몸통 파라미터 수(로그 눈금)"); ax.set_ylabel("검증 손실(나트)")
ax.set_title("② 같은 토큰 200만 개로 학습시킨 크기별 검증 손실", loc="left", fontsize=8.6, color="#374151")
ax.set_ylim(bottom=min(runs[s, D0, best[s], 0]["val"] for s in SIZES) - 0.18)    # 맨 아래 이름표가 축에 닿지 않게
ax.legend(frameon=False, fontsize=8)
for a in axes:
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_5_lr.png"); plt.close(fig)

## 5.3 크기와 토큰 수를 함께 바꾸기

In [ ]:
TOKENS = [1_000_000, 2_000_000, 4_000_000, 8_000_000]
for size in SIZES:
    for D in TOKENS:
        if (size, D, best[size], 0) not in runs:
            run(size, D, best[size])
    print(size, "  ".join(f"{D // 10**6}M:{runs[size, D, best[size], 0]['val']:.4f}({runs[size, D, best[size], 0]['sec']:.0f}초)" for D in TOKENS), flush=True)
for seed in [1, 2]:                                         # 시드만 바꿀 때 손실이 얼마나 달라지는지 한 칸에서 잰다
    run("S3", D0, best["S3"], seed)
noise = [runs["S3", D0, best["S3"], s]["val"] for s in range(3)]
print("S3, 토큰 200만 개, 시드 0·1·2:", [round(v, 4) for v in noise], f"범위 {max(noise) - min(noise):.4f}")

In [ ]:
from scipy.optimize import least_squares
rows = [(counts(LanguageModel(*SIZES[s]))[0], counts(LanguageModel(*SIZES[s]))[1], D, runs[s, D, best[s], 0]["val"], s)
        for s in SIZES for D in TOKENS]
body_n = np.array([r[0] for r in rows], float); total_n = np.array([r[1] for r in rows], float)
tokens = np.array([r[2] for r in rows], float); loss = np.array([r[3] for r in rows])

def law(p, N, D):
    """L(N, D) = E + A / N^alpha + B / D^beta. N과 D는 백만 단위로 바꿔 넣고, A와 B는 로그로 두어 양수로 맞춘다."""
    E, logA, alpha, logB, beta = p
    return E + np.exp(logA) / (N / 1e6) ** alpha + np.exp(logB) / (D / 1e6) ** beta

def fit_law(N, D, L, E=None):
    """시작점 여러 개에서 최소제곱으로 맞추고 남은 오차가 가장 작은 답을 돌려준다. E를 주면 E는 고정한다."""
    best_fit = None
    for E0 in ([1.0, 3.0] if E is None else [E]):
        for a0 in [0.1, 0.3, 0.6]:
            for b0 in [0.1, 0.3, 0.6]:
                if E is None:
                    f = lambda p: law(p, N, D) - L
                    r = least_squares(f, x0=[E0, 0.0, a0, 0.0, b0], bounds=([0, -10, 0, -10, 0], [10, 10, 3, 10, 3]))
                else:
                    f = lambda q: law([E, *q], N, D) - L
                    r = least_squares(f, x0=[0.0, a0, 0.0, b0], bounds=([-10, 0, -10, 0], [10, 3, 10, 3]))
                if best_fit is None or r.cost < best_fit.cost:
                    best_fit = r
    return best_fit.x if E is None else np.array([E, *best_fit.x])

p_body = fit_law(body_n, tokens, loss)
p_total = fit_law(total_n, tokens, loss)
rms = lambda p, N: np.sqrt(np.mean((law(p, N, tokens) - loss) ** 2))
for name, p, N in [("몸통 파라미터로 맞춘 식", p_body, body_n), ("전체 파라미터로 맞춘 식", p_total, total_n)]:
    E, logA, a, logB, b = p
    print(f"{name}: E={E:.3f}, A={math.exp(logA):.3f}, alpha={a:.3f}, B={math.exp(logB):.3f}, beta={b:.3f}, 남은 오차(RMS) {rms(p, N):.4f}")
gap = law(p_body, body_n, tokens) - loss
print("몸통 식의 칸별 차이가 가장 큰 곳:", " ".join(f"{rows[i][4]}·{rows[i][2] // 10**6}M {gap[i]:+.3f}" for i in np.argsort(-abs(gap))[:3]))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.9), dpi=260)
ax = axes[0]
TSHADE = ["#FBD7B3", "#F7B479", "#F28C28", "#B85F0F"]
grid_n = np.logspace(np.log10(body_n.min() * 0.7), np.log10(body_n.max() * 1.4), 100)
for j, D in enumerate(TOKENS):
    sel = tokens == D
    ax.plot(body_n[sel], loss[sel], "o", color=TSHADE[j], ms=4.2, label=f"토큰 {man(D)} 개")
    ax.plot(grid_n, law(p_body, grid_n, D), "-", color=TSHADE[j], lw=1.2)
ax.set_xscale("log"); param_axis(ax); ax.set_xlabel("몸통 파라미터 수(로그 눈금)"); ax.set_ylabel("검증 손실(나트)")
ax.set_title("① 크기에 따른 손실(점은 실제 학습, 선은 맞춘 식 L(N, D))", loc="left", fontsize=8.6, color="#374151")
ax.set_ylim(top=7.25)
ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", columnspacing=0.8, handletextpad=0.2)
ax = axes[1]
grid_d = np.logspace(np.log10(TOKENS[0] * 0.7), np.log10(TOKENS[-1] * 1.4), 100)
for i, s in enumerate(SIZES):
    sel = np.array([r[4] == s for r in rows])
    ax.plot(tokens[sel], loss[sel], "o", color=SHADE[i], ms=4.2, label=s)
    ax.plot(grid_d, law(p_body, body_n[sel][0], grid_d), "-", color=SHADE[i], lw=1.2)
ax.set_xscale("log"); ax.set_xticks(TOKENS); ax.xaxis.set_major_formatter(FuncFormatter(man)); ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("학습한 토큰 수(로그 눈금)"); ax.set_ylabel("검증 손실(나트)")
ax.set_title("② 토큰 수에 따른 손실(크기별, 진할수록 큰 모델)", loc="left", fontsize=8.6, color="#374151")
ax.legend(frameon=False, fontsize=8, ncol=3, columnspacing=0.8, handletextpad=0.2)
for a in axes:
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_5_grid.png"); plt.close(fig)

## 5.4 같은 계산량에서 어떤 크기가 나은가

In [ ]:
flops_6nd = 6 * body_n * tokens                                     # 흔히 쓰는 어림: 6 × 파라미터 × 토큰
dims = np.array([SIZES[r[4]][0] for r in rows], float)
flops_head = 6 * (body_n + dims * VOCAB) * tokens                    # 출력층(벡터 길이 × 어휘)까지 넣은 어림
secs = np.array([runs[r[4], r[2], best[r[4]], 0]["sec"] for r in rows])
for name, c in [("6·N_몸통·D", flops_6nd), ("6·(N_몸통 + d·V)·D", flops_head)]:
    print(f"{name}와 실제 학습 시간의 상관(로그): {np.corrcoef(np.log(c), np.log(secs))[0, 1]:.3f}")
print("가장 작은 S1의 1초당 6·N_몸통·D", f"{(flops_6nd / secs)[[r[4] == 'S1' for r in rows]].mean():.3g}",
      "가장 큰 S6", f"{(flops_6nd / secs)[[r[4] == 'S6' for r in rows]].mean():.3g}")
# 학습 시간 순서로 늘어놓고, 그 시간까지 가장 낮은 손실을 낸 학습만 고른다(시간 대비 손실의 경계)
front, low = [], float("inf")
for i in np.argsort(secs):
    if loss[i] < low:
        low = loss[i]
        front.append(i)
print("경계:", "  ".join(f"{rows[i][4]}·{rows[i][2] // 10**6}M {loss[i]:.3f}({secs[i]:.0f}초)" for i in front))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 6.3), dpi=260, gridspec_kw={"height_ratios": [1, 1.3]})
ax = axes[0]
for name, c, color in [("6·N_몸통·D", flops_6nd, RED), ("6·(N_몸통 + d·V)·D", flops_head, GREEN)]:
    r = np.corrcoef(np.log(c), np.log(secs))[0, 1]
    ax.plot(c, secs, "o", color=color, ms=3.8, alpha=0.85, label=f"{name}  (로그 상관 {r:.3f})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 10**12:g}조" if v >= 10**12 else f"{v / 10**8:g}억"))
ax.set_xlabel("어림한 계산량(곱셈·덧셈 횟수, 로그 눈금)"); ax.set_ylabel("실제 학습 시간(초)")
ax.set_yticks([20, 50, 100, 200, 400]); ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}")); ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_title("① 계산량 어림과 실제 시간\n출력층까지 넣은 초록 점이 한 줄에 더 가깝게 모인다", loc="left", fontsize=8.6, color="#374151")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax = axes[1]
for i, s in enumerate(SIZES):
    sel = np.array([r[4] == s for r in rows])
    ax.plot(secs[sel], loss[sel], "-o", color=SHADE[i], lw=1.2, ms=3.4, label=s)
ax.plot(secs[front], loss[front], "--", color=ORANGE, lw=1.6, label="그 시간까지 가장 낮은 손실")
ax.plot(secs[front], loss[front], "o", mfc="none", mec=ORANGE, mew=1.3, ms=7)
for i in front:                                                     # 경계 위의 점 몇 개에만 이름표를 단다
    if (rows[i][4], rows[i][2]) in [("S1", 1_000_000), ("S1", 2_000_000), ("S3", 4_000_000), ("S4", 8_000_000), ("S6", 8_000_000)]:
        leftmost = i == front[0]                                    # 맨 왼쪽 점은 축 눈금과 겹치지 않게 오른쪽에 단다
        ax.annotate(f"{rows[i][4]}·{man(rows[i][2])}", (secs[i], loss[i]), xytext=(8, -2) if leftmost else (0, -13),
                    textcoords="offset points", fontsize=8, color="#B85F0F", ha="left" if leftmost else "center")
ax.set_ylim(bottom=loss.min() - 0.16)
ax.set_xscale("log"); ax.set_xticks([20, 50, 100, 200, 400]); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("실제 학습 시간(초, 로그 눈금)"); ax.set_ylabel("검증 손실(나트)")
ax.set_title("② 크기마다 토큰 100만·200만·400만·800만 개의 점을 이었다", loc="left", fontsize=8.6, color="#374151")
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right", columnspacing=0.8, handletextpad=0.3)
for a in axes:
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_5_compute.png"); plt.close(fig)

## 5.5 작은 실험으로 큰 모델 맞히기

In [ ]:
# 가장 큰 S6을 빼고 맞춘 식으로 S6의 손실을 미리 맞힌다
keep = np.array([r[4] != "S6" for r in rows])
p_small = fit_law(body_n[keep], tokens[keep], loss[keep])
pred = law(p_small, body_n[~keep], tokens[~keep])
for D, real, guess in zip(tokens[~keep], loss[~keep], pred):
    print(f"S6, 토큰 {int(D) // 10**6}백만 개: 실제 {real:.4f}  예측 {guess:.4f}  차이 {guess - real:+.4f}")
E, logA, a, logB, b = p_small
print(f"S6을 뺀 식: E={E:.3f}, alpha={a:.3f}, beta={b:.3f}")
# 토큰 800만 개를 빼고 맞춘 식으로 800만 개를 맞힌다
keep_d = tokens < 8_000_000
p_short = fit_law(body_n[keep_d], tokens[keep_d], loss[keep_d])
gap = law(p_short, body_n[~keep_d], tokens[~keep_d]) - loss[~keep_d]
print("토큰 800만 개를 뺀 식의 800만 개 예측 차이:", " ".join(f"{r[4]}:{g:+.4f}" for r, g in zip(np.array(rows, dtype=object)[~keep_d], gap)))
# 바닥 E를 0~4로 못박고 24칸 전부에 다시 맞춘다. 남은 오차가 비슷하면 자료가 E를 정하지 못한다는 뜻이다
BIG_N, BIG_D = 100 * body_n.max(), 10 * tokens.max()                # S6의 100배 몸통, 토큰 8천만 개
fixed = {}
for E in [0, 1, 2, 3, 4]:
    fixed[E] = fit_law(body_n, tokens, loss, E=E)
    print(f"E={E}: alpha={fixed[E][2]:.3f}, beta={fixed[E][4]:.3f}, 남은 오차 {rms(fixed[E], body_n):.4f}, "
          f"S6·8천만 예측 {law(fixed[E], body_n.max(), BIG_D):.3f}, 100배 모델·8천만 예측 {law(fixed[E], BIG_N, BIG_D):.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 6.4), dpi=260)
ax = axes[0]
grid_n = np.logspace(np.log10(body_n.min() * 0.7), np.log10(body_n.max() * 3), 120)
for j, D in enumerate(TOKENS):
    sel = tokens == D
    ax.plot(grid_n, law(p_small, grid_n, D), "-", color=TSHADE[j], lw=1.2, label=f"토큰 {man(D)} 개")
    ax.plot(body_n[sel & keep], loss[sel & keep], "o", color=TSHADE[j], ms=3.8)
    ax.plot(body_n[sel & ~keep], loss[sel & ~keep], "s", mfc="white", mec=TSHADE[j], mew=1.4, ms=5.6)
edge = (body_n[keep].max() * body_n.max()) ** 0.5             # S5와 S6의 가운데(로그 눈금)에서 띠를 시작한다
ax.axvspan(edge, grid_n.max(), color=GRAY, alpha=0.12, lw=0)
ax.text(edge * 1.06, 6.75, "맞출 때\n쓰지 않은 범위", fontsize=8, color="#4b5563", va="top")
ax.set_xscale("log"); param_axis(ax); ax.set_xlabel("몸통 파라미터 수(로그 눈금)"); ax.set_ylabel("검증 손실(나트)")
ax.set_title("① S1~S5로 맞춘 식(선)과 S6의 실제 손실(빈 네모)", loc="left", fontsize=8.6, color="#374151")
ax.set_ylim(top=7.25)                                                # 위쪽 빈 곳에 범례를 둔다
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2, columnspacing=0.8, handlelength=1.4)
ax = axes[1]
wide = np.logspace(np.log10(body_n.min() * 0.7), np.log10(BIG_N * 1.15), 160)
ESHADE = ["#1AA37A", "#5FB98F", "#8A5CD1", "#B08AE0", "#E5484D"]
for E, color in zip(fixed, ESHADE):
    ax.plot(wide, law(fixed[E], wide, BIG_D), "-", color=color, lw=1.3, label=f"E={E}에 못박은 식(남은 오차 {rms(fixed[E], body_n):.4f})")
ax.axvspan(body_n.min() * 0.9, body_n.max() * 1.1, color=BLUE, alpha=0.08, lw=0)
ax.text(body_n.min(), ax.get_ylim()[0] + 0.03, "실험한 범위", fontsize=8, color=BLUE, va="bottom")
ax.axvline(BIG_N, color=GRAY, lw=0.8, ls=":")
ax.text(BIG_N / 1.15, ax.get_ylim()[1], "S6의 100배", fontsize=8, color="#4b5563", ha="right", va="top")
ends = [law(fixed[E], BIG_N, BIG_D) for E in fixed]
ax.annotate("", (BIG_N * 1.25, min(ends)), (BIG_N * 1.25, max(ends)), arrowprops=dict(arrowstyle="<->", color="#374151", lw=1))
ax.text(BIG_N * 1.33, (min(ends) + max(ends)) / 2, f"{max(ends) - min(ends):.3f}", fontsize=8, color="#374151", va="center")
ax.set_xscale("log"); param_axis(ax); ax.set_xlabel("몸통 파라미터 수(로그 눈금)"); ax.set_ylabel("예측 손실(나트)")
ax.set_title(f"② 토큰 {man(BIG_D)} 개일 때의 예측(바닥 E만 바꿔 맞춘 다섯 식)", loc="left", fontsize=8.6, color="#374151")
ax.set_xlim(right=BIG_N * 3)
ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0.0, 0.08))
for a in axes:
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_5_extra.png"); plt.close(fig)